# Milestone 1 — Audio Input Pipeline
### Stages 1–3 of NOVA: Microphone → Wake Word → Voice Activity Detection

This notebook teaches the concepts behind the first three stages of NOVA **before** you wire them into the real assistant. We'll cover:

1. How digital audio actually works (samples, sample rate, frames)
2. Capturing audio with `sounddevice`
3. The idea behind wake-word detection (and a mini one you build yourself)
4. Voice Activity Detection (VAD) with energy-based methods and with Silero VAD

> **Note on this environment:** this notebook runs in a sandboxed container with no microphone or speakers attached. Every cell either (a) runs fully on synthetic/generated audio so you can see the concepts work, or (b) is clearly marked as "run this on your own laptop" for the live-microphone parts. Copy those cells into a local Jupyter/VS Code session on your Windows machine to hear it live.


In [ ]:
# Install dependencies (run once)
# On your local machine (not this sandbox) run:
# pip install sounddevice numpy scipy matplotlib openwakeword onnxruntime torch soundfile

import numpy as np
import matplotlib.pyplot as plt

print("Ready. (sounddevice import is deferred to the live-mic section below)")


## 1. What *is* digital audio?

A microphone produces a continuous analog voltage. To work with it in code we **sample** it — measure its amplitude at fixed intervals.

- **Sample rate**: how many measurements per second. Speech models like Whisper expect **16,000 Hz (16 kHz)**.
- **Sample**: one amplitude measurement, usually a 16-bit integer or 32-bit float.
- **Frame / chunk**: a small batch of samples you process at once (e.g. 30 ms = 480 samples at 16kHz). Real-time audio pipelines never wait for "the whole recording" — they process frame by frame.

Let's synthesize a simple audio signal (a sine wave, like a pure tone) so you can see these ideas without needing a real mic.


In [ ]:
SAMPLE_RATE = 16000  # Hz - the standard for speech models (Whisper, Silero VAD, openWakeWord)
DURATION_S = 1.0
FREQ_HZ = 440  # A4 musical note, for illustration

t = np.linspace(0, DURATION_S, int(SAMPLE_RATE * DURATION_S), endpoint=False)
tone = 0.3 * np.sin(2 * np.pi * FREQ_HZ * t)

print(f"Total samples: {len(tone)}")
print(f"That's {DURATION_S}s of audio at {SAMPLE_RATE} Hz")

plt.figure(figsize=(10, 3))
plt.plot(t[:400], tone[:400])  # just the first 25ms so you can see the wave shape
plt.title("First 25ms of a 440Hz tone")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()


## 2. Frames and streaming

NOVA can't wait for you to finish talking before it starts processing — it listens continuously in small **frames**. Let's split our tone into 30ms frames, the way `sounddevice`'s callback API will hand you audio in real time.


In [ ]:
FRAME_MS = 30
frame_len = int(SAMPLE_RATE * FRAME_MS / 1000)
frames = [tone[i:i+frame_len] for i in range(0, len(tone), frame_len)]

print(f"Frame length: {frame_len} samples ({FRAME_MS}ms)")
print(f"Number of frames in 1 second of audio: {len(frames)}")
print(f"Shape of one frame: {frames[0].shape}")


## 3. Capturing real audio with `sounddevice`

`sounddevice` gives you two styles of API:

- **Blocking**: `sd.rec(...)` — records a fixed duration, simple but not suited to "always listening"
- **Callback / stream**: you register a function that gets called automatically with each new frame — this is what a real voice assistant needs, because it never stops listening

Below is the **actual code you'll use on your Windows machine**. It won't run in this sandbox (no audio hardware), but read it carefully — this callback pattern is the backbone of Milestone 1.


In [ ]:
# --- RUN THIS ON YOUR OWN MACHINE, NOT IN THIS SANDBOX ---
import sounddevice as sd
import numpy as np
import queue

SAMPLE_RATE = 16000
FRAME_MS = 30
FRAME_SAMPLES = int(SAMPLE_RATE * FRAME_MS / 1000)

audio_queue = queue.Queue()

def audio_callback(indata, frames, time_info, status):
    """Called automatically by sounddevice for every new audio frame."""
    if status:
        print("Stream status:", status)
    # indata is a numpy array of shape (frames, channels)
    audio_queue.put(indata.copy())

def list_input_devices():
    for i, dev in enumerate(sd.query_devices()):
        if dev["max_input_channels"] > 0:
            print(i, dev["name"])

# Uncomment to see your available microphones:
# list_input_devices()

# Open a continuous input stream
stream = sd.InputStream(
    samplerate=SAMPLE_RATE,
    channels=1,
    blocksize=FRAME_SAMPLES,
    dtype="float32",
    callback=audio_callback,
)

# Example usage:
# with stream:
#     print("Listening... Ctrl+C to stop")
#     while True:
#         frame = audio_queue.get()   # blocks until a new frame arrives
#         # ... pass `frame` into wake word / VAD here ...


### Why a queue?

The callback runs on a separate real-time audio thread. You should never do slow work (model inference, disk I/O) *inside* the callback — it will cause audio glitches. Instead, the callback's only job is to push the frame into a thread-safe `queue.Queue()`, and your main loop pulls frames off the queue at its own pace. This producer/consumer pattern is the same one you'll use for wake word + VAD + STT.


In [ ]:
# A tiny simulation of the producer/consumer pattern, fully runnable here (no mic needed)
import queue
import time

sim_queue = queue.Queue()

def fake_audio_producer(n_frames=5):
    for i in range(n_frames):
        frame = np.random.randn(frame_len).astype(np.float32) * 0.01  # fake "silence" frame
        sim_queue.put(frame)
        print(f"[producer] pushed frame {i}")

def fake_consumer(n_frames=5):
    for _ in range(n_frames):
        frame = sim_queue.get()
        print(f"[consumer] processing frame with {len(frame)} samples, energy={np.abs(frame).mean():.5f}")

fake_audio_producer()
fake_consumer()


## 4. Wake word detection — the concept

`openWakeWord` runs a small neural network continuously over a rolling window of audio, outputting a probability score for "was the wake word just spoken?" every frame. When the score crosses a threshold, you trigger listening.

Two things matter here conceptually:
1. It must run **offline** and **fast** — no network round trip, because it's checked on every single frame, all day.
2. It uses a **sliding window**, not single frames — "Hey Nova" spans many frames, so the model looks at the last ~1 second of audio each time, not just the current 30ms frame.

Let's build a *toy* wake-word detector ourselves using a much simpler technique — energy + pattern matching — just to build intuition before you use the real neural model.


In [ ]:
# Toy wake-word idea: detect a distinctive *pattern* of loud-quiet-loud energy
# (Hey Nova has a rough rhythm: HEY [pause] NO-VA)
# This is NOT how openWakeWord works internally (it uses a trained neural net on audio embeddings),
# but building this toy version teaches you *why* a sliding window over ~1s of audio is needed.

def synthetic_utterance(pattern, frame_len=frame_len, sr=SAMPLE_RATE):
    """pattern: list of ('loud'|'quiet', n_frames)"""
    audio = []
    for kind, n in pattern:
        amp = 0.4 if kind == "loud" else 0.01
        for _ in range(n):
            audio.append(np.random.randn(frame_len).astype(np.float32) * amp)
    return audio

# "Hey Nova" -> loud, quiet, loud  (very rough analogy)
hey_nova_like = synthetic_utterance([("loud", 5), ("quiet", 2), ("loud", 6)])
random_speech = synthetic_utterance([("loud", 10)])

def sliding_window_energy_pattern(frames, window=13):
    """Look at the last `window` frames and report the loud/quiet pattern."""
    energies = [np.abs(f).mean() for f in frames[-window:]]
    threshold = 0.15
    pattern = ["loud" if e > threshold else "quiet" for e in energies]
    return pattern

print("Hey-Nova-like pattern:", sliding_window_energy_pattern(hey_nova_like))
print("Random speech pattern:", sliding_window_energy_pattern(random_speech))


Notice we needed to look at a **window** of 13 frames (~390ms) to see the loud-quiet-loud shape — a single frame tells you nothing. This is exactly why `openWakeWord` buffers a rolling window of audio embeddings rather than scoring frame-by-frame in isolation.

### Real openWakeWord usage (run locally)


In [ ]:
# --- RUN THIS ON YOUR OWN MACHINE ---
from openwakeword.model import Model

# Loads a pretrained wake word model. "hey_jarvis" ships by default;
# for a custom "Hey Nova" you train/fine-tune your own model file (openWakeWord supports this).
oww_model = Model(wakeword_models=["hey_jarvis"])  # swap for your trained "hey_nova.onnx" later

def process_frame(frame_int16):
    prediction = oww_model.predict(frame_int16)  # expects int16 PCM, ~80ms chunks
    for wakeword, score in prediction.items():
        if score > 0.5:
            print(f"Wake word '{wakeword}' detected! score={score:.2f}")

# In your real audio_callback / consumer loop:
# frame_int16 = (frame * 32767).astype(np.int16)
# process_frame(frame_int16)


## 5. Voice Activity Detection (VAD)

Once woken, NOVA needs to know **when you start and stop talking** so it knows when to send audio to Whisper. Two approaches:

1. **Energy-based VAD** (simple, what we'll build below) — flag a frame as speech if its energy exceeds a threshold. Fast but easily fooled by background noise (fans, typing).
2. **Silero VAD** (what NOVA actually uses) — a small neural network trained to distinguish speech from non-speech far more robustly than raw energy.


In [ ]:
# Simple energy-based VAD you can reason about end to end
def energy_vad(frames, threshold=0.05, min_speech_frames=3):
    """Returns list of (is_speech: bool) per frame, then finds start/end."""
    flags = [np.abs(f).mean() > threshold for f in frames]
    return flags

def find_utterance_boundaries(flags, frame_ms=FRAME_MS):
    """Find the first and last speech frame -> convert to a time range."""
    speech_idxs = [i for i, f in enumerate(flags) if f]
    if not speech_idxs:
        return None
    start_ms = speech_idxs[0] * frame_ms
    end_ms = speech_idxs[-1] * frame_ms
    return start_ms, end_ms

# Build a fake utterance: silence, then speech, then silence
fake_utterance = (
    synthetic_utterance([("quiet", 5)]) +
    synthetic_utterance([("loud", 10)]) +
    synthetic_utterance([("quiet", 5)])
)

flags = energy_vad(fake_utterance)
print("Speech flags per frame:", [int(f) for f in flags])
print("Utterance boundaries (ms):", find_utterance_boundaries(flags))


### Why energy alone isn't enough

Try imagining a loud fan running throughout — every frame would read as "speech" and NOVA would never stop listening. This is exactly why production systems use **Silero VAD**, a tiny (~1MB) neural model trained on real speech vs. noise, instead of a raw amplitude threshold.


In [ ]:
# --- RUN THIS ON YOUR OWN MACHINE ---
import torch

# Silero VAD loads via torch.hub - tiny and CPU-friendly
vad_model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    force_reload=False,
)
(get_speech_timestamps, _, read_audio, *_ ) = utils

# Example on a wav file recorded from your mic:
# wav = read_audio("my_test_clip.wav", sampling_rate=16000)
# speech_timestamps = get_speech_timestamps(wav, vad_model, sampling_rate=16000)
# print(speech_timestamps)  # list of {"start": sample_idx, "end": sample_idx}


## 6. Putting Stage 1-3 together

The real pipeline you're building this milestone looks like this:

```
sounddevice callback (continuous 30ms frames)
        │
        ▼
   queue.Queue()
        │
        ▼
 openWakeWord.predict(frame)  →  score > threshold?  →  no: discard frame, keep listening
        │ yes
        ▼
 start buffering frames, run Silero VAD on each new frame
        │
        ▼
 VAD says "speech ended" (e.g. 700ms of silence)
        │
        ▼
 hand the buffered utterance audio to Whisper (Milestone 2)
```

### Exercises
1. Modify `energy_vad` above to require **3 consecutive** speech frames before triggering (reduces false positives from clicks/pops).
2. Record 5 seconds of yourself saying "Hey Nova" on your machine, save as a `.wav`, and run it through the real `openWakeWord` code above — print the confidence score over time and plot it.
3. Combine wake word + VAD into one state machine with three states: `IDLE`, `LISTENING`, `PROCESSING`. Sketch the transitions before you code it.

**Next notebook:** Milestone 2 — Speech-to-Text with `faster-whisper` (say the word and I'll build it).
